# Jailbreak Augmentation (per-stage)

Applies jailbreak technique combinations to input prompts (plan → batched execute, resumable).
Reads inputs from `{data_dir}/Datasets/` unless you pass an `inputs` DataFrame. `model=None`
uses the `uncensored_gen` role; translation routes to its own `translation` role (DeepSeek).

In [ ]:
from redact import generate_jailbreaks, build_dataset, default_escalation_schedule, set_seed
set_seed(42)

DATA_DIR = "./runs/eval"
MODEL = None
PURE_ONLY = False            # True -> no-LLM smoke run (pure transforms only)
ESCALATE = False             # True -> multi-round increasing-complexity schedule

## Generate jailbreaks

In [ ]:
jailbreaks = generate_jailbreaks(
    data_dir=DATA_DIR,
    model=MODEL,
    pure_only=PURE_ONLY,
    include_translation=False,     # drop the costly DeepSeek translation family
    entry_types=["harmful"],
    settings_per_iteration=default_escalation_schedule() if ESCALATE else None,
    batch_size=256,
    resume=True,                   # resume=False re-plans + regenerates
)
print(f"{len(jailbreaks)} jailbreak rows")
jailbreaks["technique"].value_counts().head(10)

## Complexity / round distribution (when escalating)

In [ ]:
if "iteration" in jailbreaks.columns and "num_techniques" in jailbreaks.columns:
    print(jailbreaks.groupby("iteration")["num_techniques"].mean())

## Merge everything under this data_dir

In [ ]:
dataset = build_dataset(data_dir=DATA_DIR)
dataset["dataset_type"].value_counts()